# JetBot 即時道路跟隨（TensorRT）

使用 TensorRT 優化後的模型進行即時推論，搭配 **PD 控制器** 驅動 JetBot 馬達。

## 流程
1. 載入 TensorRT 模型
2. 啟動相機即時擷取影像
3. 模型推論預測目標座標 (X, Y)
4. PD 控制器計算馬達出力
5. 即時驅動 JetBot 跟隨道路

## 1. 匯入套件

In [ ]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from torch2trt import TRTModule
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg, Robot
import numpy as np
import cv2
import time

## 2. 載入 TensorRT 模型

In [ ]:
TRT_MODEL_PATH = 'best_steering_model_xy_trt.pth'

# 載入 TensorRT 模型
model_trt = TRTModule()
model_trt.load_state_dict(torch.load(TRT_MODEL_PATH))

device = torch.device('cuda')
print(f'TensorRT 模型已載入: {TRT_MODEL_PATH} ✓')

## 3. 啟動相機 & Robot

In [ ]:
# 啟動相機
camera = Camera.instance(width=224, height=224, fps=10)

# 啟動 Robot（馬達控制）
robot = Robot()

print('相機已啟動 ✓')
print('Robot 已啟動 ✓')

## 4. 定義影像前處理

In [ ]:
# 與訓練時相同的正規化參數
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess(image):
    """
    將相機影像轉換為模型輸入 tensor
    image: BGR numpy array (224, 224, 3)
    return: CUDA tensor (1, 3, 224, 224)
    """
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.transpose((2, 0, 1))  # HWC -> CHW
    image = torch.from_numpy(image).float()
    image = image.to(device).half()  # 搬到 GPU + 半精度
    image = image / 255.0  # [0, 255] -> [0, 1]
    image = (image - mean[:, None, None]) / std[:, None, None]
    return image.unsqueeze(0)  # 加 batch 維度

print('前處理函數已定義 ✓')

## 5. 建立控制參數滑桿

In [ ]:
# 速度增益（建議初始值 0.15，避免衝出跑道）
speed_gain_slider = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.01,
    value=0.15, description='speed_gain'
)

# P 增益（比例項）
steering_pgain_slider = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.01,
    value=0.05, description='P_gain'
)

# D 增益（微分項，抑制震盪）
steering_dgain_slider = widgets.FloatSlider(
    min=0.0, max=1.0, step=0.01,
    value=0.0, description='D_gain'
)

# 馬達公差修正
steering_bias_slider = widgets.FloatSlider(
    min=-0.3, max=0.3, step=0.01,
    value=0.0, description='bias'
)

# 顯示預測結果
x_slider = widgets.FloatSlider(
    min=-1.0, max=1.0, description='pred_x'
)
y_slider = widgets.FloatSlider(
    min=-1.0, max=1.0, description='pred_y'
)
steering_slider = widgets.FloatSlider(
    min=-1.0, max=1.0, description='steering'
)

# 即時影像
image_widget = widgets.Image(format='jpeg', width=224, height=224)

display(
    image_widget,
    speed_gain_slider,
    steering_pgain_slider,
    steering_dgain_slider,
    steering_bias_slider,
    x_slider,
    y_slider,
    steering_slider
)

## 6. 定義 PD 控制器 & 執行迴圈

In [ ]:
# 全域變數：記錄上一次角度（用於 D 項計算）
angle_last = 0.0

def execute(change):
    """
    每次相機更新時執行：
    1. 前處理影像
    2. 模型推論
    3. PD 控制器計算馬達出力
    4. 驅動 JetBot
    """
    global angle_last
    
    # 取得當前相機影像
    image = change['new']
    
    # 前處理 & 推論
    input_tensor = preprocess(image)
    output = model_trt(input_tensor).detach().cpu().numpy().flatten()
    x = float(output[0])
    y = float(output[1])
    
    # 更新顯示
    x_slider.value = x
    y_slider.value = y
    
    # PD 控制器
    speed_gain = speed_gain_slider.value
    p_gain = steering_pgain_slider.value
    d_gain = steering_dgain_slider.value
    bias = steering_bias_slider.value
    
    # 計算目標角度
    angle = np.arctan2(x, y)
    
    # PD 控制
    # P 項：當前角度 * 增益
    # D 項：角度變化率 * 增益
    pid = angle * p_gain + (angle - angle_last) * d_gain
    angle_last = angle
    
    # 加入馬達公差修正
    steering = pid + bias
    steering_slider.value = steering
    
    # 計算左右馬達出力（限制在 0~1）
    left_motor = max(min(speed_gain + steering, 1.0), 0.0)
    right_motor = max(min(speed_gain - steering, 1.0), 0.0)
    
    # 驅動馬達
    robot.left_motor.value = left_motor
    robot.right_motor.value = right_motor
    
    # 更新顯示影像（繪製預測點）
    display_image = np.copy(image)
    pixel_x = int(x * 112 + 112)
    pixel_y = int(y * 112 + 112)
    display_image = cv2.circle(display_image, (pixel_x, pixel_y), 8, (0, 255, 0), 3)
    image_widget.value = bgr8_to_jpeg(display_image)

print('PD 控制器已定義 ✓')
print('\n注意：首次執行建議 speed_gain 設為 0.15')

## 7. 開始即時道路跟隨

In [ ]:
# 將 execute 函數綁定到相機更新事件
# 每次相機擷取新影像時，自動呼叫 execute
execute({'new': camera.value})
camera.observe(execute, names='value')

print('🚗 JetBot 道路跟隨已啟動！')
print('調整滑桿控制速度和轉向參數')

## 8. 停止 JetBot

In [ ]:
# 取消相機事件綁定
camera.unobserve(execute, names='value')

# 停止馬達
robot.left_motor.value = 0.0
robot.right_motor.value = 0.0

# 停止相機
time.sleep(0.1)
camera.stop()

print('JetBot 已停止 ✓')
print('相機已釋放 ✓')